# prior sign-off

quick check that the residuals and the graph are in good shape before we train the diffusion model.

two rules: everything after cell 3 runs on the *normalized* residuals, and we use the locked gamma per dataset.

three gates:
- **cell 2** train-vs-val residual scale (metr-la + pems08f must pass; pems-bay is allowed to fail)
- **cell 5** graph is one connected piece (all three must pass)
- **cell 6** laplacian lambda_2 has a clear gap (all three must pass)

everything else is just a number we read off or compute. set the dataset in cell 0 and run top to bottom; re-run for each dataset.

In [12]:
# pick the dataset and run the whole notebook. change this and re-run for the next one.
dataset = "metrla"   # metrla | pems08f | pemsbay

import os, sys
# run from the repo root so the src imports work
while not os.path.exists("pyproject.toml"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

from src.data.adjacency import compute_normalized_laplacian, load_adjacency
from src.diffusion.prior import GAMMA_LOCKED
from src.utils import paths

seed = 0
gamma = GAMMA_LOCKED[dataset]
headline = dataset in ("metrla", "pems08f")   # these have to pass the scale gate
print("dataset =", dataset, " gamma =", round(gamma, 6), " headline =", headline)

dataset = metrla  gamma = 0.157555  headline = True


### cell 1 — load and eyeball
load the residuals (B, N, H) and the adjacency, and confirm they look sane.

In [13]:
def load_resid(split):
    d = np.load(paths.residual_npz(dataset, split, seed))
    return d["residuals"].astype(np.float64), d["mask"].astype(bool)

R_tr, M_tr = load_resid("train")
R_vl, M_vl = load_resid("val")

A_raw = load_adjacency(dataset).astype(np.float64)
A = 0.5 * (A_raw + A_raw.T)   # symmetrize

B, N, H = R_tr.shape
print("residuals train:", R_tr.shape, " val:", R_vl.shape)
print("adjacency:", A.shape, " N =", N, " H =", H, " B_train =", B)
print("train mask coverage:", round(M_tr.mean(), 4), "(1.0 = nothing masked)")
print("one window R[0,0,:] =", np.round(R_tr[0, 0], 2))

residuals train: (23967, 207, 12)  val: (3405, 207, 12)
adjacency: (207, 207)  N = 207  H = 12  B_train = 23967
train mask coverage: 0.9287 (1.0 = nothing masked)
one window R[0,0,:] = [-2.17 -4.63  0.58  3.85 -6.93  1.61  1.27  0.63 -0.26 -0.81  1.    2.82]


### cell 2 — gate 1: train-vs-val scale
per-horizon residual scale should match between train and val. delta_h is the masked mean of the squared residual at horizon h. pems-bay fails here by design.

In [14]:
def per_horizon_scale(R, M):
    return np.array([(R[:, :, h][M[:, :, h]] ** 2).mean() for h in range(H)])

delta_tr = per_horizon_scale(R_tr, M_tr)
delta_vl = per_horizon_scale(R_vl, M_vl)
ratio = delta_vl / delta_tr - 1.0
scale_ok = bool((np.abs(ratio) < 0.10).all())

print(" h   delta_tr    delta_vl    vl/tr-1")
for h in range(H):
    mark = "ok" if abs(ratio[h]) < 0.10 else "XX"
    print(f"{h+1:>2}  {delta_tr[h]:>9.3f}  {delta_vl[h]:>9.3f}  {ratio[h]:>+7.2%}  {mark}")

if headline:
    print("gate 1 (must pass):", "PASS" if scale_ok else "FAIL")
else:
    print("gate 1 (pems-bay limitation, fail expected):", "noted" if not scale_ok else "PASS")

 h   delta_tr    delta_vl    vl/tr-1
 1     14.859     14.232   -4.22%  ok
 2     21.869     20.675   -5.46%  ok
 3     27.365     25.666   -6.21%  ok
 4     32.025     29.934   -6.53%  ok
 5     36.005     33.682   -6.45%  ok
 6     39.512     37.025   -6.29%  ok
 7     42.647     40.090   -6.00%  ok
 8     45.487     42.884   -5.72%  ok
 9     48.117     45.471   -5.50%  ok
10     50.550     47.792   -5.46%  ok
11     52.972     50.048   -5.52%  ok
12     55.624     52.479   -5.65%  ok
gate 1 (must pass): PASS


### cell 3 — normalize
divide by one scalar so every measurement below is on the same scale. keep sigma_r — we need it to de-normalize the samples at inference.

In [15]:
sigma_r = float(R_tr[M_tr].std())   # masked std over the train residuals
Rn = R_tr / sigma_r
print("sigma_r =", round(sigma_r, 6))

sigma_r = 6.20323


### cell 4 — temporal structure
build Sigma_T from the normalized residuals. drop any row with a masked horizon, then fit D_H (robust per-horizon variance) and R_H (horizon correlation). mu are the eigenvalues of Sigma_T.

In [16]:
flat = Rn.reshape(B * N, H)
keep = M_tr.reshape(B * N, H).all(axis=1)
Rk = flat[keep]

mad = np.median(np.abs(Rk - np.median(Rk, axis=0)), axis=0)
D_H = (1.4826 * mad) ** 2
R_H = np.corrcoef(Rk.T)

d = np.sqrt(D_H)
Sigma_T = (d[:, None] * R_H) * d[None, :]
mu = np.clip(np.linalg.eigvalsh(Sigma_T), 1e-12, None)
mu_max, mu_min = float(mu[-1]), float(mu[0])

print("D_H   :", np.round(D_H, 3))
print("R_H[0]:", np.round(R_H[0], 3))
print("mu    :", np.round(mu, 4))
print("mu_max =", round(mu_max, 4), " mu_min =", round(mu_min, 4))

D_H   : [0.089 0.097 0.104 0.109 0.114 0.119 0.123 0.127 0.131 0.136 0.14  0.145]
R_H[0]: [1.    0.482 0.36  0.295 0.25  0.218 0.195 0.174 0.157 0.144 0.132 0.122]
mu    : [0.0205 0.0223 0.0245 0.0272 0.0306 0.0354 0.0423 0.053  0.071  0.1091
 0.2171 0.7815]
mu_max = 0.7815  mu_min = 0.0205


### cell 5 — gate 2: connectivity
the sensor graph has to be one connected piece. if not, it prints the stranded sensors — fix those in the adjacency, don't paper over it with gamma.

In [17]:
adj = (A != 0).astype(int)
np.fill_diagonal(adj, 0)   # self-loops don't change connectivity
n_comp, label = connected_components(csr_matrix(adj), directed=False)
conn_ok = n_comp == 1

print("connected components:", n_comp)
if not conn_ok:
    sizes = np.bincount(label)
    main = sizes.argmax()
    print("component sizes:", sizes.tolist())
    print("stranded sensors:", np.where(label != main)[0].tolist())
print("gate 2 (must pass):", "PASS" if conn_ok else "FAIL")

connected components: 2
component sizes: [206, 1]
stranded sensors: [26]
gate 2 (must pass): FAIL


### cell 6 — gate 3: laplacian gap
lambda_1 should be ~0 with a clear gap up to lambda_2. a tiny lambda_2 means weak clusters.

note: this gate and cell 5 catch different things. under the symmetric-normalized laplacian an *isolated* sensor (degree 0) shows up as an eigenvalue of 1, not 0 — so it hides in the middle of the spectrum and this gate won't see it. that's why cell 5 (the integer component count) is the one that catches stranded sensors.

In [18]:
L = compute_normalized_laplacian(A_raw)   # strips diag + symmetrizes inside
ev = np.linalg.eigvalsh(L)
lam2, lam_max = float(ev[1]), float(ev[-1])

print("smallest 8 eigenvalues:", np.round(ev[:8], 5))
print("lambda_1 =", f"{ev[0]:.2e}", " lambda_2 =", round(lam2, 5), " lambda_max =", round(lam_max, 5))
gap_ok = abs(ev[0]) < 1e-6 and lam2 > 1e-3
print("gate 3 (lambda_2 clearly above 0):", "PASS" if gap_ok else "REVIEW")

smallest 8 eigenvalues: [-0.       0.00757  0.01196  0.01722  0.03452  0.06862  0.07696  0.14247]
lambda_1 = -6.09e-16  lambda_2 = 0.00757  lambda_max = 1.70621
gate 3 (lambda_2 clearly above 0): PASS


### cell 7 — check Sigma_S is positive definite
Sigma_S = inv(L + gamma I); its eigenvalues are just 1/(lambda_k + gamma), so we read them off without forming the inverse. min eigenvalue must be > 0 and the condition number finite.

In [19]:
sig_s = 1.0 / (ev + gamma)
print("Sigma_S min eigenvalue:", f"{sig_s.min():.4e}", "(must be > 0)")
print("Sigma_S condition number:", round((lam_max + gamma) / (ev[0] + gamma), 2))

Sigma_S min eigenvalue: 5.3655e-01 (must be > 0)
Sigma_S condition number: 11.83


### cell 8 — the schedule numbers
the precision eigenvalues are (lambda_k + gamma) / mu_h. Theta_min uses lambda~0 and mu_max; Theta_max uses lambda_max and mu_min. if c0 > ~25, gamma is a touch small — bump it and re-run from cell 6.

In [20]:
theta_min = gamma / mu_max
theta_max = (lam_max + gamma) / mu_min
stiffness = theta_max / theta_min
c0 = 5.0 / theta_min

print("Theta_min =", round(theta_min, 4))
print("Theta_max =", round(theta_max, 4))
print("stiffness =", round(stiffness, 4))
print("c0        =", round(c0, 4))
if c0 > 25:
    print("warning: c0 > 25, bump gamma a notch and re-run from cell 6")

Theta_min = 0.2016
Theta_max = 90.9742
stiffness = 451.2336
c0        = 24.8001


### cell 9 — isotropic arm
for the isotropic prior everything is flat, recorded here so both arms live in one place.

In [21]:
print("isotropic: Theta = 1, stiffness = 1, c0 = 5")

isotropic: Theta = 1, stiffness = 1, c0 = 5


### cell 10 — summary row to STATE.md
write one provenance row for this dataset. re-running the same dataset just replaces its row. three clean rows = cleared for diffusion.

In [22]:
row = {
    "dataset": dataset,
    "sigma_r": round(sigma_r, 4),
    "gamma": round(gamma, 4),
    "lam2": round(lam2, 4),
    "mu_max": round(mu_max, 4),
    "theta_min": round(theta_min, 4),
    "c0_kron": round(c0, 4),
    "c0_iso": 5.0,
    "stiffness": round(stiffness, 4),
    "scale": "PASS" if scale_ok else ("noted" if not headline else "FAIL"),
    "components": "PASS" if conn_ok else "FAIL",
    "lam2_gap": "PASS" if gap_ok else "REVIEW",
}

cols = list(row)
header = "| " + " | ".join(cols) + " |"
sep = "|" + "|".join(["---"] * len(cols)) + "|"
line = "| " + " | ".join(str(row[c]) for c in cols) + " |"

state = paths.ROOT / "STATE.md"
lines = state.read_text().splitlines() if state.exists() else []
if not lines:
    lines = [header, sep]
lines = [l for l in lines if not l.startswith(f"| {dataset} ")]   # drop old row for this dataset
lines.append(line)
state.write_text("\n".join(lines) + "\n")

print("updated", state)
print(line)

updated /Users/serkan/Projects/research-tud/STATE.md
| metrla | 6.2032 | 0.1576 | 0.0076 | 0.7815 | 0.2016 | 24.8001 | 5.0 | 451.2336 | PASS | FAIL | PASS |
